# HDB Price Prediction Regression Model

In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error, r2_score,
)
import numpy as np

## 1. Analyse Dataset

In [2]:
# Load dataset
df = pd.read_csv('HDB_Resale_Prices.csv')

# Display first few rows
df.head(5)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2025-05,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,04 TO 06,44.0,Improved,1979,53 years 01 month,340000.0
1,2025-08,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,07 TO 09,44.0,Improved,1979,52 years 10 months,353000.0
2,2025-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,52 years 02 months,318000.0
3,2025-03,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,51 years 11 months,338000.0
4,2025-09,ANG MO KIO,3 ROOM,308A,ANG MO KIO AVE 1,13 TO 15,70.0,Model A,2012,86 years,625000.0


In [3]:
# Get random samples
df.sample(5)

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
22843,2025-07,WOODLANDS,5 ROOM,782B,WOODLANDS CRES,07 TO 09,112.0,Improved,2015,89 years,670000.0
12445,2025-10,PASIR RIS,4 ROOM,639,PASIR RIS DR 1,01 TO 03,104.0,Model A,1995,68 years 10 months,560000.0
20149,2025-08,TAMPINES,5 ROOM,609B,TAMPINES NTH DR 1,01 TO 03,113.0,Improved,2020,94 years 03 months,830000.0
10372,2025-01,JURONG WEST,4 ROOM,458,JURONG WEST ST 41,10 TO 12,91.0,New Generation,1984,58 years 10 months,500000.0
12665,2025-12,PASIR RIS,4 ROOM,718,PASIR RIS ST 72,01 TO 03,111.0,Model A,1996,69 years 08 months,647000.0


In [4]:
# Dataset shape (rows, columns)
print(f'(rows, columns): {df.shape}')

# Column names
print(f'Columns: {df.columns.tolist()}')

print('\n--- DATASET INFO ---')
df.info()

print('\n--- DATASET DESCRIPTION ---')
df.describe()

(rows, columns): (25094, 11)
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']

--- DATASET INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 25094 entries, 0 to 25093
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                25094 non-null  str    
 1   town                 25094 non-null  str    
 2   flat_type            25094 non-null  str    
 3   block                25094 non-null  str    
 4   street_name          25094 non-null  str    
 5   storey_range         25094 non-null  str    
 6   floor_area_sqm       25094 non-null  float64
 7   flat_model           25094 non-null  str    
 8   lease_commence_date  25094 non-null  int64  
 9   remaining_lease      25094 non-null  str    
 10  resale_price         25094 non-null  float64
dtypes: float64(2), int64(1), s

,floor_area_sqm,lease_commence_date,resale_price
count,25094.000000,25094.000000,2.509400e+04
mean,94.890611,1998.340201,6.525051e+05
std,23.865320,15.325162,2.041269e+05
min,31.000000,1966.000000,2.500000e+05
25%,74.000000,1985.000000,5.100000e+05
50%,93.000000,1998.000000,6.280000e+05
75%,111.000000,2015.000000,7.600000e+05
max,195.000000,2021.000000,1.658888e+06


In [5]:
# Check for null values
print('\n--- NULL VALUES CHECK ---')
df.isnull()

# Count null values per column
df.isnull().sum()

# Percentage of missing values
print('\n--- PERCENTAGE OF MISSING VALUES ---')
print((df.isnull().sum() / len(df)) * 100)



--- NULL VALUES CHECK ---

--- PERCENTAGE OF MISSING VALUES ---
month                  0.0
town                   0.0
flat_type              0.0
block                  0.0
street_name            0.0
storey_range           0.0
floor_area_sqm         0.0
flat_model             0.0
lease_commence_date    0.0
remaining_lease        0.0
resale_price           0.0
dtype: float64


As ```storey_range``` is a string eg '01 to 03', '04 to 06, we can encode it into 1, 2, 3

-> I realised this is not required as we can simply one-hot encode everything

In [6]:
# First, check all unique values
# unique_storeys = df["storey_range"].unique()

# Create automatic mapping based on the first number
# df["storey_encoded"] = df["storey_range"].str.extract(r"(\d+)").astype(int) // 3 + 1

In [7]:
# Check that encoding worked
# df[['storey_range', 'storey_encoded']].head(10)

Convert ```remaining_lease_years``` into float

In [8]:
# Extract years and months separately
df["years"] = df["remaining_lease"].str.extract(r"(\d+) years").astype(float)
df["months"] = df["remaining_lease"].str.extract(r"(\d+) month").astype(float)

# Fill NaN values with 0 (for cases with only years or only months)
df["years"] = df["years"].fillna(0)
df["months"] = df["months"].fillna(0)

# Convert to decimal years
df["remaining_lease_float"] = df["years"] + (df["months"] / 12)

# Check the new columns
df[["remaining_lease", "years", "months", "remaining_lease_float"]].head(10)

,remaining_lease,years,months,remaining_lease_float
0,53 years 01 month,53.0,1.0,53.083333
1,52 years 10 months,52.0,10.0,52.833333
2,52 years 02 months,52.0,2.0,52.166667
3,51 years 11 months,51.0,11.0,51.916667
4,86 years,86.0,0.0,86.000000
5,51 years 04 months,51.0,4.0,51.333333
6,51 years 02 months,51.0,2.0,51.166667
7,51 years 05 months,51.0,5.0,51.416667
8,50 years 07 months,50.0,7.0,50.583333
9,60 years 02 months,60.0,2.0,60.166667


In [9]:
print("\n--- DATASET INFO (After Processing) ---")
df.info()


--- DATASET INFO (After Processing) ---
<class 'pandas.DataFrame'>
RangeIndex: 25094 entries, 0 to 25093
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   month                  25094 non-null  str    
 1   town                   25094 non-null  str    
 2   flat_type              25094 non-null  str    
 3   block                  25094 non-null  str    
 4   street_name            25094 non-null  str    
 5   storey_range           25094 non-null  str    
 6   floor_area_sqm         25094 non-null  float64
 7   flat_model             25094 non-null  str    
 8   lease_commence_date    25094 non-null  int64  
 9   remaining_lease        25094 non-null  str    
 10  resale_price           25094 non-null  float64
 11  years                  25094 non-null  float64
 12  months                 25094 non-null  float64
 13  remaining_lease_float  25094 non-null  float64
dtypes: float64(5), int64(1),

## Model Training

In [13]:
y = df['resale_price']

feature_columns = ['town', 'flat_type', 'storey_range', 'floor_area_sqm', 'flat_model', 'remaining_lease_float']

X = df[feature_columns]

In [16]:
# Categorical features (need encoding)
categorical_features = ["town", "flat_type", "flat_model", "storey_range"]

# Numerical features (already in numeric form)
numerical_features = ["floor_area_sqm", "remaining_lease_years"]

In [17]:
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

print(f"Original shape: {X.shape}")
print(f"After encoding: {X_encoded.shape}")

Original shape: (25094, 6)
After encoding: (25094, 69)


## Train-test-validation split

In [20]:
# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_encoded, y, test_size=0.15, random_state=42
)

# Second split: separate validation set from remaining data (15% of total = ~18% of temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.15/0.85 ≈ 0.176
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Training set: 17575 samples
Validation set: 3754 samples
Test set: 3765 samples


## Train

| Performance Level | R² Score    | Expected for HDB           |
| ----------------- | ----------- | -------------------------- |
| Poor              | < 0.70      | Missing key features       |
| Fair              | 0.70 - 0.80 | Basic model                |
| Good              | 0.80 - 0.90 | Solid performance          |
| Excellent         | 0.90 - 0.95 | Very Strong performance        |
| Exceptional       | > 0.95      | Rare, possible overfitting |

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

# Make predictions
y_val_pred = rf_model.predict(X_val)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
mape = mean_absolute_percentage_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print(f"RMSE: ${rmse:,.2f}")
print(f"MAE: ${mae:,.2f}")
print("MAPE %:", mape * 100)
print(f"R² Score: {r2:.4f}")

RMSE: $55,116.07
MAE: $36,138.50
MAPE %: 5.3913995788490325
R² Score: 0.9256


### Custom Input

In [59]:
sample_input = {
    "town": "WOODLANDS",
    "flat_type": "5 ROOM",
    "storey_range": "04 TO 06",
    "floor_area_sqm": 112.0,
    "flat_model": "Improved",
    "remaining_lease_years": 86.8333333333,
}

input_df = pd.DataFrame([sample_input])

input_encoded = pd.get_dummies(input_df, columns=categorical_features, drop_first=True)

# align to training feature columns (missing dummies become 0; extra cols dropped)
input_encoded = input_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Make prediction
predicted_price = rf_model.predict(input_encoded)[0]

print(f"Predicted resale price: ${predicted_price:,.2f}")

Predicted resale price: $721,998.83
